# Step 3 — verify-then-report (thin notebook: it only calls scripts)
**Settings:** GPU T4 x2, Internet on, Secret `HF_TOKEN` ticked.
**Inputs:** the Step 1/2 datasets (Indiana, ReXGradient test images, train subset), **`organizations/nih-chest-xrays/data`**
(only its expert boxes are used, to measure box quality), and the outputs of your finished Step 0, 1 and 2 notebooks.

Each generated sentence -> CheXbert finding + side -> evidence from the Step 2 Probe and MedGemma 1.5 boxes ->
VERIFIED / UNCERTAIN / FLAGGED (rules in `src/verify/verdict.py`). Acceptance: share of clinically wrong sentences
caught, and clinical metrics after dropping FLAGGED sentences.
If Step 2 KEPT retrieval, first set `verify.source` in `configs/step3.yaml` to the retrieval system.

In [ ]:
!git clone -q --depth 1 https://github.com/Vinith-44/Radioreport /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q -r requirements.txt

In [ ]:
import os, json
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["RUN_NOTE"] = "step3 kaggle"

## 0. Restore Step 0-2 results (and a previous Step 3 session, to resume)

In [ ]:
PREV = []   # e.g. ["/kaggle/input/step0-kaggle/project", "/kaggle/input/step1-kaggle/project",
            #       "/kaggle/input/step2-kaggle/project"]  (+ a previous step3 output to resume)
os.makedirs("/kaggle/working/project", exist_ok=True)
for src in PREV:
    !cp -rn {src}/outputs /kaggle/working/project/
    !cp -rn {src}/data /kaggle/working/project/ 2>/dev/null || true
!ls /kaggle/working/project/outputs/runs

## 1. Rebuild the parquet files (test subset must be identical to Step 0), run the tests

In [ ]:
ids_p = "/kaggle/working/project/data/step0_eval_ids.json"
before = json.load(open(ids_p)) if os.path.exists(ids_p) else None
!python scripts/step0_prepare_data.py --config configs/step3.yaml
assert before is None or before == json.load(open(ids_p)), "test eval ids changed - comparisons would be void"
!python -m pytest tests -q

## 2. Smoke run (needs the Step 1/2 smoke runs in the restored outputs)

In [ ]:
!python scripts/step0_prepare_data.py --config configs/step3.yaml --smoke
!python scripts/step3_claims.py --run-name v1 --smoke
!python scripts/step3_localize.py --run-name v1 --smoke
!python scripts/step3_verify.py --run-name v1 --smoke
!python scripts/step3_score.py --run-name v1 --smoke
!python scripts/step3_eval_boxes.py --run-name v1 --smoke

## 3. Box quality on NIH expert boxes (~320 queries) - look at this BEFORE trusting box evidence
MedGemma's documented localisation (38 IoU) is for anatomy; boxes for findings are what this measures.

In [ ]:
!python scripts/step3_eval_boxes.py --run-name v1

## 4. Full verification (box queries are the slow part: resumable, re-run after a timeout)
To bound the time, set `verify.max_studies_per_dataset` (e.g. 1000) in `configs/step3.yaml` before step3_claims.

In [ ]:
!python scripts/step3_claims.py --run-name v1
!python scripts/step3_localize.py --run-name v1

In [ ]:
!python scripts/step3_verify.py --run-name v1
!python scripts/step3_score.py --run-name v1
!cat /kaggle/working/project/outputs/RESULTS.md